# NB03 — Data Splits (70/10/20 + held-outs)

From `benchmark_cleaned.csv` (unique, `label5`), build:
1. **Random 70/10/20** train/val/test, stratified on `label5`. Test (20%) is the reported holdout.
2. **Source-held-out** (×4): test = one unseen source, train+val = the other three.
3. **Script-held-out** (×2): test = one unseen script, train+val = the other.

Every split writes CSVs to `data/splits/` and asserts **zero overlap** between any train/val and its test.


In [ ]:
import os, json
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
SEED = 42
DATA = "../data/processed/benchmark_cleaned.csv"
SPLIT_DIR = "../data/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)
df = pd.read_csv(DATA).reset_index(drop=True)
df["uid"] = df.index
LABEL = "label5"
assert LABEL in df.columns and df[LABEL].nunique() == 5
print(f"Loaded {len(df):,} rows | classes: {sorted(df[LABEL].unique())}")
print("Distribution:", df[LABEL].value_counts().to_dict())

In [ ]:
# ── 1) Random 70/10/20 stratified on label5 ────────────────────────────────
train_df, tmp = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df[LABEL])
val_df, test_df = train_test_split(tmp, test_size=2/3, random_state=SEED, stratify=tmp[LABEL])
# 0.30 → val 0.10, test 0.20  (test_size=2/3 of the 30% = 20%)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(f"{SPLIT_DIR}/random_{name}.csv", index=False, encoding="utf-8-sig")
print(f"random  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print(f"        ratios  {len(train_df)/len(df):.2f}/{len(val_df)/len(df):.2f}/{len(test_df)/len(df):.2f}")

# leakage check
tr_val_ids = set(train_df["uid"]) | set(val_df["uid"])
assert len(tr_val_ids & set(test_df["uid"])) == 0, "LEAK: test overlaps train/val"
print("✅ no overlap between (train∪val) and test")

print("\nPer-class counts across splits:")
print(pd.concat([train_df[LABEL].value_counts().rename("train"),
                 val_df[LABEL].value_counts().rename("val"),
                 test_df[LABEL].value_counts().rename("test")], axis=1).to_string())

In [ ]:
# ── 2) Source-held-out splits (×4) ──────────────────────────────────────────
# test = ALL rows of the held-out source; train/val = stratified 88/12 of the rest.
sources = sorted(df["source"].unique())
src_manifest = {}
for src in sources:
    test_s = df[df["source"] == src]
    rest   = df[df["source"] != src]
    tr_s, val_s = train_test_split(rest, test_size=0.125, random_state=SEED, stratify=rest[LABEL])
    tag = f"source_holdout_{src}"
    tr_s.to_csv(f"{SPLIT_DIR}/{tag}_train.csv", index=False, encoding="utf-8-sig")
    val_s.to_csv(f"{SPLIT_DIR}/{tag}_val.csv", index=False, encoding="utf-8-sig")
    test_s.to_csv(f"{SPLIT_DIR}/{tag}_test.csv", index=False, encoding="utf-8-sig")
    # HARD leakage guard
    seen = set(tr_s["uid"]) | set(val_s["uid"])
    leaked = len(seen & set(test_s["uid"]))
    assert leaked == 0, f"LEAK in {tag}: {leaked} rows"
    src_manifest[src] = {"train": len(tr_s), "val": len(val_s), "test": len(test_s)}
    print(f"{tag:32s} train={len(tr_s):6,} val={len(val_s):5,} test={len(test_s):6,}  ✅ leak=0")

In [ ]:
# ── 3) Script-held-out splits (×2) ──────────────────────────────────────────
scripts = sorted(df["script"].unique())
scr_manifest = {}
for scr in scripts:
    test_s = df[df["script"] == scr]
    rest   = df[df["script"] != scr]
    # stratify only if the rest has >=2 of each class; else fall back to no stratify
    strat = rest[LABEL] if rest[LABEL].value_counts().min() >= 2 else None
    tr_s, val_s = train_test_split(rest, test_size=0.125, random_state=SEED, stratify=strat)
    tag = f"script_holdout_{scr}"
    tr_s.to_csv(f"{SPLIT_DIR}/{tag}_train.csv", index=False, encoding="utf-8-sig")
    val_s.to_csv(f"{SPLIT_DIR}/{tag}_val.csv", index=False, encoding="utf-8-sig")
    test_s.to_csv(f"{SPLIT_DIR}/{tag}_test.csv", index=False, encoding="utf-8-sig")
    seen = set(tr_s["uid"]) | set(val_s["uid"])
    leaked = len(seen & set(test_s["uid"]))
    assert leaked == 0, f"LEAK in {tag}: {leaked} rows"
    scr_manifest[scr] = {"train": len(tr_s), "val": len(val_s), "test": len(test_s)}
    print(f"{tag:32s} train={len(tr_s):6,} val={len(val_s):5,} test={len(test_s):6,}  ✅ leak=0")

In [ ]:
# ── Manifest ────────────────────────────────────────────────────────────────
manifest = {
    "seed": SEED, "label_col": LABEL, "n_total": len(df),
    "random": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "source_holdout": src_manifest, "script_holdout": scr_manifest,
}
with open(f"{SPLIT_DIR}/split_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("✅ saved split_manifest.json")
print(json.dumps(manifest, indent=2))